In [1]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

sp500 = pd.read_html(url, storage_options=headers)[0]
sp500['Symbol'] = sp500['Symbol'].str.replace('.','-')

symbols_list = sp500['Symbol'].unique().tolist()

end_date = '2026-10-10'

start_date = pd.to_datetime(end_date) - pd.DateOffset(years=8)

download = yf.download(
    tickers=symbols_list,
    start=start_date,
    end=end_date,
    auto_adjust=False
)
df = download.stack(level=1)
df.index.names = ['date', 'ticker']
df.columns = df.columns.str.lower()

df


[*********************100%***********************]  503 of 503 completed


Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-10-10 A        62.858780   66.669998   69.410004   66.610001   69.410004   
           AAPL     51.112507   54.090000   56.587502   54.012501   56.365002   
           ABBV     65.991730   93.080002   95.389999   92.989998   93.849998   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      59.256882   68.919998   71.209999   68.779999   71.050003   
...                       ...         ...         ...         ...         ...   
2026-10-09 XYZ      77.239998   77.239998   77.680000   75.260002   75.625000   
           YUM     144.820007  144.820007  145.470001  143.039993  144.000000   
           ZBH      89.139999   89.139999   90.209999   88.400002   88.930000   
           ZBRA    387.179993  387.179993  389.010010  381.989990  386.459991   
           ZTS      74.809998   74.809998   74.849998   73.000000   73.040001   

Price                   volume  
date       ticker               
2018-10-10 A         3055700.0  
           AAPL    167962400.0  
           ABBV      5310800.0  
           ABNB            NaN  
           ABT      10559200.0  
...                        ...  
2026-10-09 XYZ       5232500.0  
           YUM       3138600.0  
           ZBH       1604300.0  
           ZBRA       338400.0  
           ZTS       2912100.0  

[1011030 rows x 6 columns]

In [2]:
df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*((np.log(df['adj close'])-np.log(df['open']))**2)

df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.rsi(close=x, length=20))

df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])
df['bb_mid'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])
df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])

def compute_atr(stock_data):
    atr = pandas_ta.atr(high=stock_data['high'],low=stock_data['low'],close=stock_data['close'],length=14)
    return atr.sub(atr.mean()).div(atr.std())

df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

# ---------- MACD (normalized) ----------
def compute_macd(close):
    try:
        m = pandas_ta.macd(close=close, fast=12, slow=26, signal=9)
        if m is None or not isinstance(m, pd.DataFrame):
            raise ValueError("macd returned None")
        macd_line = m.iloc[:, 0]
        std = macd_line.std()
        if std == 0 or pd.isna(std):
            return pd.Series(index=close.index, dtype=float)
        return (macd_line - macd_line.mean()) / std
    except Exception:
        return pd.Series(index=close.index, dtype=float)
        
df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

df['dollar_volume'] = (df['adj close']*df['volume'])/1e6

df

Price               adj close       close        high         low        open  \
date       ticker                                                               
2018-10-10 A        62.858780   66.669998   69.410004   66.610001   69.410004   
           AAPL     51.112507   54.090000   56.587502   54.012501   56.365002   
           ABBV     65.991730   93.080002   95.389999   92.989998   93.849998   
           ABNB           NaN         NaN         NaN         NaN         NaN   
           ABT      59.256882   68.919998   71.209999   68.779999   71.050003   
...                       ...         ...         ...         ...         ...   
2026-10-09 XYZ      77.239998   77.239998   77.680000   75.260002   75.625000   
           YUM     144.820007  144.820007  145.470001  143.039993  144.000000   
           ZBH      89.139999   89.139999   90.209999   88.400002   88.930000   
           ZBRA    387.179993  387.179993  389.010010  381.989990  386.459991   
           ZTS      74.809998   74.809998   74.849998   73.000000   73.040001   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2018-10-10 A         3055700.0         -0.002949        NaN       NaN   
           AAPL    167962400.0         -0.002612        NaN       NaN   
           ABBV      5310800.0         -0.047585        NaN       NaN   
           ABNB            NaN               NaN        NaN       NaN   
           ABT      10559200.0         -0.012123        NaN       NaN   
...                        ...               ...        ...       ...   
2026-10-09 XYZ       5232500.0          0.000328  49.552661  4.300954   
           YUM       3138600.0          0.000129  52.416424  4.902887   
           ZBH       1604300.0          0.000203  42.850914  4.462786   
           ZBRA       338400.0          0.000164  65.556435  5.824392   
           ZTS       2912100.0          0.000092  54.404678  4.245182   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2018-10-10 A            NaN       NaN       NaN       NaN     192.077574  
           AAPL         NaN       NaN       NaN       NaN    8584.979323  
           ABBV         NaN       NaN       NaN       NaN     350.468878  
           ABNB         NaN       NaN       NaN       NaN            NaN  
           ABT          NaN       NaN       NaN       NaN     625.705265  
...                     ...       ...       ...       ...            ...  
2026-10-09 XYZ     4.300954  4.300954 -0.776512 -0.217093     404.158289  
           YUM     4.902887  4.902887  0.918231 -0.936477     454.532075  
           ZBH     4.462786  4.462786 -0.471613 -0.806666     143.007301  
           ZBRA    5.824392  5.824392  0.177489  0.973600     131.021710  
           ZTS     4.245182  4.245182 -1.620997 -0.164591     217.854194  

[1011030 rows x 14 columns]

In [10]:
# last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open', 'high', 'low', 'close']]


data = (pd.concat([df.unstack('ticker')['dollar_volume'].resample('ME').mean().stack('ticker').to_frame('dollar_volume'),
df.unstack()[last_cols].resample('ME').last().stack('ticker')],axis=1)).dropna()

data

dollar_volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                           
2018-11-30 A          187.326249   68.214073         -0.000757  50.392546   
           AAPL      8333.096440   42.334629         -0.001346  32.688884   
           ABBV       482.619652   67.550407         -0.031305  48.837536   
           ABT        390.192304   63.929420         -0.008036  56.904926   
           ACGL        28.335316   27.214716         -0.000698  36.877469   
...                          ...         ...               ...        ...   
2026-10-31 XYZ        334.708773   77.239998          0.000328  49.552661   
           YUM        338.325318  144.820007          0.000129  52.416424   
           ZBH        182.891983   89.139999          0.000203  42.850914   
           ZBRA       158.953923  387.179993          0.000164  65.556435   
           ZTS        315.109929   74.809998          0.000092  54.404678   

                     bb_low    bb_mid   bb_high       atr      macd  
date       ticker                                                    
2018-11-30 A       4.083388  4.083388  4.083388 -1.308589  0.525551  
           AAPL    3.695726  3.695726  3.695726 -1.210623 -1.161273  
           ABBV    4.082265  4.082265  4.082265 -0.278752  0.081328  
           ABT     4.080795  4.080795  4.080795 -0.820343  0.394919  
           ACGL    3.308384  3.308384  3.308384 -1.399050 -0.026961  
...                     ...       ...       ...       ...       ...  
2026-10-31 XYZ     4.300954  4.300954  4.300954 -0.776512 -0.217093  
           YUM     4.902887  4.902887  4.902887  0.918231 -0.936477  
           ZBH     4.462786  4.462786  4.462786 -0.471613 -0.806666  
           ZBRA    5.824392  5.824392  5.824392  0.177489  0.973600  
           ZTS     4.245182  4.245182  4.245182 -1.620997 -0.164591  

[47178 rows x 9 columns]

In [15]:
# data['dollar_volume'] = (data['dollar_volume'].unstack('ticker').rolling(5*12).mean().stack())

# data['dollar_volume_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

data = data[data['dollar_volume']<150].drop(['dollar_volume','dollar_volume_rank'],axis=1)

data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2023-10-31 ACGL     82.423882         -0.000007  62.466130  4.343093   
           AEE      69.075340         -0.002859  45.402693  4.212577   
           AES      13.007282         -0.006975  49.295956  2.443970   
           AFL      73.288307         -0.001108  53.160563  4.273062   
           AIZ     142.222122         -0.000209  56.989313  4.923788   
...                       ...               ...        ...       ...   
2026-10-31 UHS     177.350006          0.000191  54.595592  5.155493   
           VMRK     61.160000          0.000216  45.516630  4.074323   
           VTRS     17.639999          0.000433  57.115798  2.865432   
           WRB      71.910004          0.000076  60.048215  4.208861   
           WY       19.100000          0.000375  31.074509  2.919396   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2023-10-31 ACGL    4.343093  4.343093  1.034814  0.645331  
           AEE     4.212577  4.212577  0.041558 -0.603476  
           AES     2.443970  2.443970 -0.182339 -0.369919  
           AFL     4.273062  4.273062 -0.132437 -0.107848  
           AIZ     4.923788  4.923788 -0.679500  0.168628  
...                     ...       ...       ...       ...  
2026-10-31 UHS     5.155493  5.155493  0.302588  0.200533  
           VMRK    4.074323  4.074323 -0.485285 -1.233009  
           VTRS    2.865432  2.865432  0.013013  0.675204  
           WRB     4.208861  4.208861  1.162530  0.269602  
           WY      2.919396  2.919396 -0.969362 -2.177064  

[5430 rows x 8 columns]